# 05 - Safeguards: Calibration, Corruption Sweep and Input-Quality Gate

**Goal:** turn the finished FP32 classifier (`best_model.keras`) into something safe to put in front of a farmer, *without retraining it*.

What this notebook measures (nothing here changes notebooks 01-04 or any existing script):
1. How well calibrated the model's confidence is (ECE, reliability table).
2. A confidence threshold chosen on the **validation** split only (coverage vs accuracy).
3. A controlled corruption sweep with explicit severities (noise, blur, darkness).
4. How severe the corruption used in `evaluate.py` actually is.
5. An input-quality gate calibrated on clean validation images, then tested on the test split.
6. The combined system (gate + threshold) with the threshold frozen.

Outputs: `advisory/gate_config.json` and `advisory/safeguards_report.json`, which notebook 06 and the app load.

In [ ]:
# Cell 1 - locate the repo root and make the existing scripts importable
import os, sys, json, time, math, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
ROOT = Path.cwd().resolve()
while not (ROOT / 'manifest.csv').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'manifest.csv').exists(), 'Open this notebook from inside the repo (manifest.csv not found)'
os.chdir(ROOT)  # manifest paths are relative to the repo root
sys.path.insert(0, str(ROOT / 'scripts'))
print('[setup] repo root :', ROOT)
print('[setup] scripts   :', ROOT / 'scripts')

In [ ]:
# Cell 2 - imports (existing modules are imported as-is, never copied)
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import albumentations as A
from dataclasses import asdict
import data_pipeline as dp
import inference_utils as iu
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)
print('[versions] TF', tf.__version__, '| albumentations', A.__version__, '| pandas', pd.__version__, '| numpy', np.__version__)
print('[classes ] n =', len(dp.SELECTED_CLASSES))

In [ ]:
# Cell 3 - experiment configuration
SEED = 42
SWEEP_PER_CLASS = 10         # images per class used in the slower corruption experiments
TARGET_SELECTIVE_ACC = 0.95  # accuracy wanted on the answers the system actually gives
MIN_COVERAGE = 0.60          # ...while still answering at least this share of images
FALLBACK_THRESHOLD = 0.80    # used only if no threshold meets the target on VAL
np.random.seed(SEED)
print('[config] sweep images per class:', SWEEP_PER_CLASS)
print('[config] target selective accuracy:', TARGET_SELECTIVE_ACC, '| min coverage:', MIN_COVERAGE)

## 1. Data audit from the manifest (single source of truth)

In [ ]:
# Cell 4 - load manifest, normalise Windows-style paths, drop rows whose file is not on this machine
manifest = pd.read_csv('manifest.csv')
manifest['File_Path'] = manifest['File_Path'].str.replace(chr(92), '/', regex=False)
manifest['on_disk'] = manifest['File_Path'].map(os.path.exists)
print('[manifest] rows:', len(manifest))
print(manifest.groupby('split')['on_disk'].agg(on_disk='sum', total='count'))
print('[manifest] files missing on disk:', int((~manifest['on_disk']).sum()))
manifest = manifest[manifest['on_disk']].drop(columns='on_disk').reset_index(drop=True)

In [ ]:
# Cell 5 - class x split table and the val / test frames used below
split_table = pd.crosstab(manifest['Class_Name'], manifest['split'])
split_table = split_table[[c for c in ['train', 'val', 'test'] if c in split_table.columns]]
split_table['total'] = split_table.sum(axis=1)
print(split_table)
val_df = manifest[manifest['split'] == 'val'].reset_index(drop=True)
test_df = manifest[manifest['split'] == 'test'].reset_index(drop=True)
assert len(val_df) > 0 and len(test_df) > 0, 'val/test images are not on disk'
print('[data] val images:', len(val_df), '| test images:', len(test_df))

In [ ]:
# Cell 6 - load the trained FP32 model (the INT8 TFLite model is NOT used: 60.9% accuracy is not deployable)
model = tf.keras.models.load_model('best_model.keras')
print('[model] input :', model.input_shape)
print('[model] outputs (concepts, diseases):', [o.shape[-1] for o in model.outputs])
print('[model] parameters:', format(model.count_params(), ','))

## 2. Clean predictions, errors and confidence calibration

In [ ]:
# Cell 7 - predict val and test once (clean); every later analysis reuses these frames
GATE_DEFAULT = iu.GateConfig()  # provisional gate; recalibrated in section 5
t0 = time.time()
val_clean = iu.predict_frame(model, val_df, cfg=GATE_DEFAULT)
test_clean = iu.predict_frame(model, test_df, cfg=GATE_DEFAULT)
print('[predict] images:', len(val_clean) + len(test_clean), '| seconds:', round(time.time() - t0))
print('[accuracy] val :', round(val_clean['correct'].mean(), 4))
print('[accuracy] test:', round(test_clean['correct'].mean(), 4), '(committed report: 0.9276)')
print(val_clean.head())

In [ ]:
# Cell 8 - per-class accuracy and the most frequent confusions on the test split
per_class = test_clean.groupby('Class_Name')['correct'].agg(n='count', accuracy='mean').sort_values('accuracy')
print(per_class.round(3))
test_clean['pred_class'] = test_clean['pred_idx'].map(lambda i: dp.SELECTED_CLASSES[i])
errors = test_clean[~test_clean['correct']]
print('[errors] wrong predictions:', len(errors))
print(errors.groupby(['Class_Name', 'pred_class']).size().sort_values(ascending=False).head(10))

In [ ]:
# Cell 9 - is confidence informative? compare confidence of correct vs wrong answers
by_outcome = test_clean.groupby('correct')['confidence'].describe().rename(index={False: 'wrong', True: 'correct'})
print(by_outcome.round(3))
print('[insight] wrong answers with confidence >= 0.9:', int((errors['confidence'] >= 0.9).sum()), 'of', len(errors))
print('[insight] mean margin (top1 - top2), correct vs wrong:')
print(test_clean.groupby('correct')['margin'].mean().rename(index={False: 'wrong', True: 'correct'}).round(3))

In [ ]:
# Cell 10 - calibration: expected calibration error and a reliability table (validation split)
ece_val = iu.expected_calibration_error(val_clean['confidence'], val_clean['correct'])
ece_test = iu.expected_calibration_error(test_clean['confidence'], test_clean['correct'])
print('[calibration] ECE val =', round(ece_val, 4), '| test =', round(ece_test, 4))
bins = pd.cut(val_clean['confidence'], bins=[0, 0.5, 0.7, 0.9, 0.99, 1.0], include_lowest=True)
reliability = val_clean.groupby(bins, observed=True).agg(n=('correct', 'size'), mean_conf=('confidence', 'mean'), accuracy=('correct', 'mean'))
print(reliability.round(3))

In [ ]:
# Cell 11 - choose the confidence threshold on VALIDATION only (test stays untouched until section 6)
cov_val = iu.coverage_accuracy_table(val_clean)
print(cov_val.round(3).to_string(index=False))
THRESHOLD = iu.pick_threshold(cov_val, TARGET_SELECTIVE_ACC, MIN_COVERAGE)
if THRESHOLD is None:
    THRESHOLD = FALLBACK_THRESHOLD
    print('[threshold] no VAL threshold met the target; using fallback', THRESHOLD)
else:
    print('[threshold] chosen on VAL only:', THRESHOLD)

In [ ]:
# Cell 12 - coverage vs selective accuracy curve (validation)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(cov_val['coverage'], cov_val['selective_accuracy'], marker='o')
ax.set_xlabel('coverage (share of images answered)')
ax.set_ylabel('accuracy on answered images')
ax.set_title('Validation: coverage vs selective accuracy')
ax.grid(alpha=0.3)
plt.show()

## 3. Controlled corruption sweep (explicit severities)
Severities are in gray levels / pixels so the result does not depend on any library default.

In [ ]:
# Cell 13 - class-balanced sample of the test split for the slower experiments
sweep_df = test_df.sample(frac=1, random_state=SEED).groupby('Class_Name').head(SWEEP_PER_CLASS).reset_index(drop=True)
clean_on_sweep = test_clean[test_clean['File_Path'].isin(sweep_df['File_Path'])]
print('[sweep] images:', len(sweep_df), '| classes:', sweep_df['Class_Name'].nunique())
print('[sweep] clean accuracy on this sample:', round(clean_on_sweep['correct'].mean(), 4))

In [ ]:
# Cell 14 - run the sweep: one corruption factor at a time
grid = ([('noise_sigma', v) for v in [0, 5, 10, 15, 25, 40, 60]] +
        [('blur_len', v) for v in [5, 9, 15]] +
        [('brightness_delta', v) for v in [-30, -60, -90]])
records = []
t0 = time.time()
for factor, value in grid:
    frame = iu.predict_frame(model, sweep_df, cfg=GATE_DEFAULT, corrupt={factor: value})
    acc = frame['correct'].mean()
    records.append({'factor': factor, 'severity': value, 'accuracy': acc, 'mean_conf': frame['confidence'].mean(),
                    'est_noise_sigma': frame['noise_sigma'].mean(), 'sharpness': frame['sharpness'].mean(),
                    'gate_pass_rate': frame['gate_passed'].mean()})
    print(f'[sweep] {factor}={value}: accuracy={acc:.3f}')
sweep_results = pd.DataFrame(records)
print('[sweep] seconds:', round(time.time() - t0))

In [ ]:
# Cell 15 - tables and curves per factor
factors = ['noise_sigma', 'blur_len', 'brightness_delta']
for factor in factors:
    print('[sweep] factor:', factor)
    print(sweep_results[sweep_results['factor'] == factor].drop(columns='factor').round(3).to_string(index=False))
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, factor in zip(axes, factors):
    sub = sweep_results[sweep_results['factor'] == factor]
    ax.plot(sub['severity'], sub['accuracy'], marker='o', label='accuracy')
    ax.plot(sub['severity'], sub['mean_conf'], marker='s', linestyle='--', label='mean confidence')
    ax.set_title(factor)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.3)
    ax.legend()
plt.show()

## 4. How severe is the corruption used in `evaluate.py`?
The committed robustness result (about 7.9% accuracy) comes from `A.GaussNoise(p=1.0)` with library defaults. Those defaults changed between albumentations 1.x and 2.x, so the *severity* depends on the installed version. This section measures it instead of assuming it.

In [ ]:
# Cell 16 - re-create the evaluate.py-style corruption and measure its real noise level
import inspect
param = inspect.signature(A.GaussNoise.__init__).parameters.get('std_range')
print('[albumentations]', A.__version__, '| GaussNoise std_range parameter:', param)
eval_like = A.Compose([A.MotionBlur(blur_limit=7, p=1.0), A.GaussNoise(p=1.0),
                       A.RandomBrightnessContrast(brightness_limit=-0.3, contrast_limit=0.3, p=1.0)])
def eval_like_corrupt(img):
    return eval_like(image=img)['image']
eval_like_frame = iu.predict_frame(model, sweep_df, cfg=GATE_DEFAULT, corrupt=eval_like_corrupt)
acc_eval_like = eval_like_frame['correct'].mean()
est = eval_like_frame['noise_sigma'].mean()
clean_med = val_clean['noise_sigma'].median()
noise_rows = sweep_results[sweep_results['factor'] == 'noise_sigma'].reset_index(drop=True)
nearest = noise_rows.iloc[(noise_rows['est_noise_sigma'] - est).abs().argsort().iloc[0]]
print('[evaluate.py-like] accuracy on sample      :', round(acc_eval_like, 3))
print('[evaluate.py-like] estimated noise sigma   :', round(est, 1), '(clean validation median:', round(clean_med, 1), ')')
print('[evaluate.py-like] noise vs clean          :', round(est / clean_med, 1), 'x')
print('[evaluate.py-like] nearest sweep severity  : sigma =', nearest['severity'], '-> sweep accuracy', round(nearest['accuracy'], 3))
if est > 3 * clean_med:
    print('[finding] this corruption is far noisier than the mild noise the model tolerates in the sweep;')
    print('          the committed ~7.9% measures extreme noise, so report it with its severity, not as general fragility.')
else:
    print('[finding] this corruption is within the range of the sweep; the committed result reflects real fragility at that level.')

## 5. Input-quality gate: calibrate on clean validation, test on the test split

In [ ]:
# Cell 17 - calibrate gate thresholds from clean VAL percentiles (with a safety margin) and save them
q = val_clean[['sharpness', 'noise_sigma', 'brightness']].quantile([0.005, 0.01, 0.5, 0.99, 0.995])
print(q.round(1))
gate_cfg = iu.GateConfig(
    min_side=100,
    min_sharpness=float(val_clean['sharpness'].quantile(0.01) * 0.75),
    max_noise_sigma=float(val_clean['noise_sigma'].quantile(0.99) * 1.25),
    min_brightness=float(val_clean['brightness'].quantile(0.005) * 0.8),
    max_brightness=float(min(250.0, val_clean['brightness'].quantile(0.995) * 1.1)))
print('[gate] calibrated config:', gate_cfg)
def apply_gate(frame):
    out = frame.copy()
    out['gate_passed'] = [iu.evaluate_gate({'sharpness': r.sharpness, 'noise_sigma': r.noise_sigma, 'brightness': r.brightness}, 224, gate_cfg)[0]
                          for r in out.itertuples()]
    return out
for name, frame in [('val', val_clean), ('test', test_clean)]:
    rate = 1 - apply_gate(frame)['gate_passed'].mean()
    print('[gate] false-reject rate on clean', name, ':', round(100 * rate, 2), '%')
gate_cfg.save()
print('[gate] saved to', iu.GATE_CONFIG_PATH)

In [ ]:
# Cell 18 - does the gate separate clean photos from degraded ones? (test sample, gate frozen)
conditions = {
    'clean': clean_on_sweep.copy(),
    'mild (noise 8, blur 5, dark 30)': iu.predict_frame(model, sweep_df, cfg=gate_cfg, corrupt={'noise_sigma': 8, 'blur_len': 5, 'brightness_delta': -30}),
    'noise 15': iu.predict_frame(model, sweep_df, cfg=gate_cfg, corrupt={'noise_sigma': 15}),
    'noise 25': iu.predict_frame(model, sweep_df, cfg=gate_cfg, corrupt={'noise_sigma': 25}),
    'blur 15': iu.predict_frame(model, sweep_df, cfg=gate_cfg, corrupt={'blur_len': 15}),
    'evaluate.py-like': eval_like_frame.copy(),
}
rows = []
for name, fr in conditions.items():
    fr = apply_gate(fr)
    accepted = fr[fr['gate_passed']]
    rejected = fr[~fr['gate_passed']]
    rows.append({'condition': name, 'n': len(fr), 'accuracy_all': fr['correct'].mean(),
                 'gate_reject_rate': 1 - fr['gate_passed'].mean(),
                 'accuracy_on_accepted': accepted['correct'].mean() if len(accepted) else np.nan,
                 'accuracy_on_rejected': rejected['correct'].mean() if len(rejected) else np.nan})
gate_report = pd.DataFrame(rows)
print(gate_report.round(3).to_string(index=False))

## 6. Combined system on the test split (threshold frozen from section 2)

In [ ]:
# Cell 19 - answer only when the photo passes the gate AND confidence >= threshold
final_rows = []
for name, fr in conditions.items():
    fr = apply_gate(fr)
    answered = fr[fr['gate_passed'] & (fr['confidence'] >= THRESHOLD)]
    final_rows.append({'condition': name, 'n': len(fr), 'unfiltered_accuracy': fr['correct'].mean(),
                       'answered_share': len(answered) / len(fr),
                       'accuracy_on_answered': answered['correct'].mean() if len(answered) else np.nan})
final_report = pd.DataFrame(final_rows)
print('[system] threshold =', THRESHOLD)
print(final_report.round(3).to_string(index=False))
tc = apply_gate(test_clean)
answered_full = tc[tc['gate_passed'] & (tc['confidence'] >= THRESHOLD)]
print('[system] FULL clean test split: answered', round(100 * len(answered_full) / len(tc), 1), '% of images,',
      'accuracy on answered =', round(answered_full['correct'].mean(), 4), '(unfiltered', round(tc['correct'].mean(), 4), ')')

In [ ]:
# Cell 20 - save the report that notebook 06 and the app load, then print a summary
report = {
    'threshold': float(THRESHOLD),
    'target_selective_accuracy': TARGET_SELECTIVE_ACC,
    'min_coverage': MIN_COVERAGE,
    'val_ece': float(ece_val), 'test_ece': float(ece_test),
    'clean_test_accuracy': float(test_clean['correct'].mean()),
    'clean_test_answered_share': float(len(answered_full) / len(tc)),
    'clean_test_accuracy_on_answered': float(answered_full['correct'].mean()),
    'gate_config': asdict(gate_cfg),
    'albumentations_version': A.__version__,
    'evaluate_like_estimated_noise_sigma': float(est),
    'clean_val_median_noise_sigma': float(clean_med),
    'gate_report': gate_report.round(4).to_dict('records'),
    'system_report': final_report.round(4).to_dict('records'),
}
out_path = ROOT / 'advisory' / 'safeguards_report.json'
with open(out_path, 'w') as f:
    json.dump(report, f, indent=2)
print('[saved]', out_path)
print('[summary] threshold            :', report['threshold'])
print('[summary] ECE (val / test)     :', round(report['val_ece'], 4), '/', round(report['test_ece'], 4))
print('[summary] clean test answered  :', round(100 * report['clean_test_answered_share'], 1), '% at accuracy', round(report['clean_test_accuracy_on_answered'], 4))
print('[summary] gate config          :', report['gate_config'])